# Первый CV-прототип: классификация пикселей водохранилища

**Задача.** По спектральным каналам одного пикселя определить один из трёх классов:
`clean_water`, `bloom`, `artifact`.

Это модель обнаружения цветения на уже имеющемся снимке, а не прогноз на 7 дней.
Её выходом позже будут карта вероятности и доля площади цветения — признаки для модели прогноза.

**Проверка.** Мы не смешиваем случайные пиксели одной картинки между train и test. На каждой итерации целая спутниковая сцена остаётся в тесте (`leave-one-scene-out`). Это существенно честнее, но пока данных всего 10 сцен и все они выбраны как сцены с цветением. Метрики ниже — техническая проверка прототипа, не окончательная оценка системы.

In [ ]:
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

warnings.filterwarnings('ignore', category=UserWarning)
sns.set_theme(style='whitegrid', context='notebook')

ROOT = Path.cwd()
DATA_PATH = ROOT / 'cv_pixel_training_dataset.csv'
AUDIT_PATH = ROOT / 'cv_pixel_dataset_audit.csv'
FEATURES = ['blue', 'green', 'red', 'nir', 'swir1', 'fai', 'mndwi', 'ndvi']
CLASS_NAMES = {0: 'clean_water', 1: 'bloom', 2: 'artifact'}
CLASS_ORDER = [0, 1, 2]
RANDOM_STATE = 42
MAX_TRAIN_PER_SCENE_CLASS = 300

## 1. Загружаем подготовленные CV-пиксели

In [ ]:
pixels = pd.read_csv(DATA_PATH)
audit = pd.read_csv(AUDIT_PATH)

print(f'Пикселей: {len(pixels):,}')
print(f'Сцен: {pixels.scene_id.nunique()}')
display(audit)
display(
    pixels.groupby(['sensor', 'class_name']).size()
    .rename('points').reset_index()
    .pivot(index='sensor', columns='class_name', values='points')
    .fillna(0).astype(int)
)

**Контроль качества.** Красные строки ниже означают, что на сцене мало примеров чистой воды. Такие сцены оставлены в прототипе, но их полезно дополнительно обвести и выгрузить заново.

In [ ]:
audit_view = audit.copy()
audit_view['needs_more_clean_water'] = audit_view['clean_water_points'] < 100
display(
    audit_view.style.apply(
        lambda row: ['background-color: #ffe5e5'] * len(row)
        if row['needs_more_clean_water'] else [''] * len(row),
        axis=1,
    )
)

## 2. Почему две модели

Sentinel-2 и Landsat измеряют похожие, но не идентичные спектральные диапазоны и имеют разное разрешение. Поэтому на первом прототипе обучаем отдельный Random Forest для каждого сенсора. Это защищает от ложной уверенности, когда модель запоминает различие спутников вместо различия воды и цветения.

In [ ]:
def make_model():
    return RandomForestClassifier(
        n_estimators=180,
        max_depth=14,
        min_samples_leaf=4,
        class_weight='balanced_subsample',
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def evaluate_by_scene(sensor_df: pd.DataFrame):
    '''Leave-one-scene-out predictions for one sensor.'''
    oof_parts = []
    scene_rows = []
    for held_out_scene in sorted(sensor_df['scene_id'].unique()):
        train_all = sensor_df[sensor_df.scene_id != held_out_scene]
        # One broad polygon must not outweigh every other scene. Keep
        # at most 300 pixels from each scene and class in each fold.
        balanced_parts = []
        for _, group in train_all.groupby(['scene_id', 'class']):
            balanced_parts.append(group.sample(
                n=min(len(group), MAX_TRAIN_PER_SCENE_CLASS),
                random_state=RANDOM_STATE,
            ))
        train = pd.concat(balanced_parts, ignore_index=True)
        test = sensor_df[sensor_df.scene_id == held_out_scene].copy()
        model = make_model().fit(train[FEATURES], train['class'])
        test['predicted_class'] = model.predict(test[FEATURES])
        probabilities = model.predict_proba(test[FEATURES])
        for class_value in CLASS_ORDER:
            position = list(model.classes_).index(class_value)
            test[f'p_{CLASS_NAMES[class_value]}'] = probabilities[:, position]
        present_classes = sorted(test['class'].unique())
        scene_rows.append({
            'scene_id': held_out_scene,
            'requested_date': test['requested_date'].iloc[0],
            'sensor': test['sensor'].iloc[0],
            'test_points': len(test),
            'classes_present_in_test': ', '.join(CLASS_NAMES[item] for item in present_classes),
            'accuracy': accuracy_score(test['class'], test['predicted_class']),
            'balanced_accuracy_present_classes': balanced_accuracy_score(test['class'], test['predicted_class']),
            'macro_f1_present_classes': f1_score(
                test['class'], test['predicted_class'], labels=present_classes,
                average='macro', zero_division=0,
            ),
        })
        oof_parts.append(test)
    return pd.concat(oof_parts, ignore_index=True), pd.DataFrame(scene_rows)

## 3. Честная проверка: новая целая сцена для теста

In [ ]:
all_oof = []
all_scene_metrics = []
for sensor, sensor_pixels in pixels.groupby('sensor', sort=True):
    if sensor_pixels.scene_id.nunique() < 2:
        print(f'{sensor}: пропуск — нужно как минимум две сцены.')
        continue
    oof, scene_metrics = evaluate_by_scene(sensor_pixels)
    all_oof.append(oof)
    all_scene_metrics.append(scene_metrics)

oof_predictions = pd.concat(all_oof, ignore_index=True)
scene_metrics = pd.concat(all_scene_metrics, ignore_index=True)
display(scene_metrics.sort_values(['sensor', 'requested_date']))

overall_metrics = []
for sensor, frame in oof_predictions.groupby('sensor', sort=True):
    overall_metrics.append({
        'sensor': sensor,
        'oof_points': len(frame),
        'oof_scenes': frame.scene_id.nunique(),
        'accuracy': accuracy_score(frame['class'], frame['predicted_class']),
        'balanced_accuracy': balanced_accuracy_score(frame['class'], frame['predicted_class']),
        'macro_f1': f1_score(frame['class'], frame['predicted_class'], labels=CLASS_ORDER, average='macro', zero_division=0),
    })
overall_metrics = pd.DataFrame(overall_metrics)
display(overall_metrics)

In [ ]:
fig, axes = plt.subplots(1, len(overall_metrics), figsize=(7 * len(overall_metrics), 5), squeeze=False)
for ax, sensor in zip(axes.ravel(), overall_metrics['sensor']):
    frame = oof_predictions[oof_predictions.sensor == sensor]
    matrix = confusion_matrix(frame['class'], frame['predicted_class'], labels=CLASS_ORDER, normalize='true')
    sns.heatmap(
        matrix, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
        xticklabels=[CLASS_NAMES[item] for item in CLASS_ORDER],
        yticklabels=[CLASS_NAMES[item] for item in CLASS_ORDER], ax=ax,
    )
    ax.set_title(f'{sensor}: OOF confusion matrix')
    ax.set_xlabel('Предсказанный класс')
    ax.set_ylabel('Истинный ручной класс')
plt.tight_layout()
plt.show()

## 3.1 Как CV работает на невидимой сцене

Ниже показана последняя по времени сцена каждого сенсора, которую модель не видела при обучении на соответствующей итерации. Слева — ручные полигоны, в центре — класс, выбранный моделью, справа — вероятность класса `bloom`.

Так же будет обрабатываться любой новый спутниковый снимок: для каждого пикселя рассчитываются 8 признаков, Random Forest выдаёт три вероятности, а классом становится наиболее вероятный из них.

In [ ]:
colors = {0: '#2196F3', 1: '#43A047', 2: '#FB8C00'}
labels = {0: 'clean_water', 1: 'bloom', 2: 'artifact'}
fig, axes = plt.subplots(len(overall_metrics), 3, figsize=(15, 5 * len(overall_metrics)), squeeze=False)

for row_index, sensor in enumerate(sorted(overall_metrics['sensor'])):
    candidates = oof_predictions[oof_predictions.sensor == sensor]
    demo_scene_id = candidates.sort_values('requested_date').scene_id.iloc[-1]
    frame = candidates[candidates.scene_id == demo_scene_id]
    title_prefix = f'{sensor} | {frame.requested_date.iloc[0]} | {len(frame):,} test pixels'

    for class_value, class_label in labels.items():
        subset = frame[frame['class'] == class_value]
        axes[row_index, 0].scatter(
            subset.longitude, subset.latitude, s=7, alpha=0.7,
            color=colors[class_value], label=class_label,
        )
    axes[row_index, 0].set_title(title_prefix + ': ручная разметка')
    axes[row_index, 0].legend(markerscale=2, loc='best')

    for class_value, class_label in labels.items():
        subset = frame[frame['predicted_class'] == class_value]
        axes[row_index, 1].scatter(
            subset.longitude, subset.latitude, s=7, alpha=0.7,
            color=colors[class_value], label=class_label,
        )
    axes[row_index, 1].set_title(title_prefix + ': прогноз CV')
    axes[row_index, 1].legend(markerscale=2, loc='best')

    probability_plot = axes[row_index, 2].scatter(
        frame.longitude, frame.latitude, c=frame.p_bloom,
        s=8, alpha=0.8, cmap='YlGn', vmin=0, vmax=1,
    )
    axes[row_index, 2].set_title(title_prefix + ': вероятность bloom')
    fig.colorbar(probability_plot, ax=axes[row_index, 2], label='P(bloom)')
    for axis in axes[row_index]:
        axis.set_xlabel('longitude')
        axis.set_ylabel('latitude')
        axis.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.show()

## 4. Итоговые модели и важность признаков

Эти модели обучаются на всех доступных пикселях и сохраняются для применения к новым снимкам. Важность признака показывает, насколько он помогал разделять классы в этом конкретном наборе. Это не причинный вывод о природе цветения.

In [ ]:
importances = []
full_models = {}
for sensor, sensor_pixels in pixels.groupby('sensor', sort=True):
    model = make_model().fit(sensor_pixels[FEATURES], sensor_pixels['class'])
    full_models[sensor] = model
    model_path = ROOT / f'cv_pixel_model_{sensor.lower()}.joblib'
    joblib.dump({'model': model, 'features': FEATURES, 'class_names': CLASS_NAMES}, model_path)
    for feature, importance in zip(FEATURES, model.feature_importances_):
        importances.append({'sensor': sensor, 'feature': feature, 'importance': importance})

importances = pd.DataFrame(importances)
fig, axes = plt.subplots(1, len(full_models), figsize=(7 * len(full_models), 5), squeeze=False)
for ax, sensor in zip(axes.ravel(), sorted(full_models)):
    frame = importances[importances.sensor == sensor].sort_values('importance')
    ax.barh(frame.feature, frame.importance, color='#1976D2')
    ax.set_title(f'{sensor}: importance')
    ax.set_xlabel('Random Forest importance')
plt.tight_layout()
plt.show()
display(importances.sort_values(['sensor', 'importance'], ascending=[True, False]))

## 5. Сохраняем результаты и ограничения

In [ ]:
oof_predictions.to_csv(ROOT / 'cv_pixel_oof_predictions.csv', index=False)
scene_metrics.to_csv(ROOT / 'cv_pixel_scene_metrics.csv', index=False)
overall_metrics.to_csv(ROOT / 'cv_pixel_model_metrics.csv', index=False)
importances.to_csv(ROOT / 'cv_pixel_feature_importance.csv', index=False)

print('Saved:')
for path in [
    ROOT / 'cv_pixel_oof_predictions.csv',
    ROOT / 'cv_pixel_scene_metrics.csv',
    ROOT / 'cv_pixel_model_metrics.csv',
    ROOT / 'cv_pixel_feature_importance.csv',
]:
    print('-', path.name)

print(
    '\nInterpretation limits:\n'
    '- 10 scenes are enough for a technical prototype, not for a final accuracy claim.\n'
    '- All current scenes were initially selected as bloom references; add clean-water and review-first scenes before presenting final metrics.\n'
    '- The next practical step is to apply each sensor model to held-out imagery in GEE and calculate bloom area fraction per scene.'
)